# Central Virginia Tree Canopy — Build `smap_df` and `canopy_df`

This notebook assembles the two DataFrames that `1_sagemaker_export.py` expects:

| DataFrame | Source files | Used for |
|:---|:---|:---|
| `smap_df` | `smap_annual_mean_2015.csv` … `smap_annual_mean_2023.csv` | SMAP soil moisture time-series chart |
| `canopy_df` | `all_2016_centroids.csv` + `*_chm.tif` files | Tree canopy cover comparison bar chart |

**Run all cells in order**, then run `1_sagemaker_export.py` (or paste its cells below) to generate and upload the charts.

## Cell 1 — Configuration
Edit the S3 paths here if your bucket layout differs.

In [ ]:
import boto3
import io
import os
import glob

import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer

# ── S3 bucket and key prefixes ────────────────────────────────────────────────
BUCKET            = "central-virginia-tree-canopy-project"
SMAP_S3_PREFIX    = "data/smap/"          # location of smap_annual_mean_YYYY.csv files
CHM_S3_PREFIX     = "data/chm/charlottesville/"  # location of *_chm.tif files
CENTROID_S3_KEY   = "data/outputs/Charlottesville/all_2016_centroids.csv"
SUMMARY_S3_KEY    = "data/smap/smap_annual_summary.csv"

# ── Local working directory on the notebook instance ─────────────────────────
WORK_DIR = "/home/sagemaker-user/tree_canopy_data"
os.makedirs(WORK_DIR, exist_ok=True)

# ── Year range covered by SMAP data ──────────────────────────────────────────
SMAP_YEARS = list(range(2015, 2024))   # 2015 – 2023 inclusive

# ── CRS constants (from the LiDAR processing fix) ────────────────────────────
FT_TO_M    = 0.3048006096              # US survey foot → metres
LIDAR_CRS  = "EPSG:2284"               # NAD83 / Virginia State Plane North, US ft
WGS84_CRS  = "EPSG:4326"

s3 = boto3.client("s3", region_name="us-east-1")
print("Configuration loaded.")

---
## Part A — Build `smap_df`

### Cell 2 — Download and Concatenate All Annual SMAP CSVs

Each `smap_annual_mean_YYYY.csv` has columns: `year, lat, lon, sm_mean_m3m3, valid_days`.  
We stack all years into a single long-format DataFrame and add a `date` column (Jan 1 of each year)  
so Plotly's `px.line` can use it as a time axis.

In [ ]:
smap_frames = []

for year in SMAP_YEARS:
    s3_key   = f"{SMAP_S3_PREFIX}smap_annual_mean_{year}.csv"
    local_fp = f"{WORK_DIR}/smap_annual_mean_{year}.csv"

    try:
        s3.download_file(BUCKET, s3_key, local_fp)
        df_year = pd.read_csv(local_fp)
        smap_frames.append(df_year)
        print(f"  {year} : {len(df_year):>4} pixels  (s3://{BUCKET}/{s3_key})")
    except Exception as e:
        print(f"  {year} : SKIPPED — {e}")

# Stack all years
smap_all = pd.concat(smap_frames, ignore_index=True)

# Add a proper date column (Jan 1 of each year) for time-series plotting
smap_all["date"] = pd.to_datetime(smap_all["year"].astype(str) + "-01-01")

print(f"\nTotal SMAP pixel-year records : {len(smap_all):,}")
smap_all.head()

### Cell 3 — Assign Each SMAP Pixel to a County / Jurisdiction

The export script uses `color="county"` in `px.line`, so each row needs a `county` label.  
We use a simple bounding-box lookup based on the pixel's `lat`/`lon`.  
Adjust the bounding boxes below to match your 8-jurisdiction study area.

In [ ]:
# Bounding boxes for each jurisdiction (lat_min, lat_max, lon_min, lon_max)
# Extend or edit this dict to cover all 8 jurisdictions in your study area.
JURISDICTION_BBOX = {
    "Charlottesville" : (38.00, 38.10, -78.54, -78.43),
    "Albemarle"       : (37.85, 38.28, -78.75, -78.10),
    "Greene"          : (38.27, 38.46, -78.55, -78.25),
    "Fluvanna"        : (37.75, 38.00, -78.40, -78.00),
    "Louisa"          : (37.75, 38.05, -78.00, -77.65),
    "Nelson"          : (37.60, 37.95, -79.20, -78.65),
    "Augusta"         : (37.75, 38.40, -79.35, -78.85),
    "Rockingham"      : (38.25, 38.60, -79.20, -78.60),
}

def assign_county(lat: float, lon: float) -> str:
    """Return the jurisdiction name for a given lat/lon, or 'Other'."""
    for name, (lat_min, lat_max, lon_min, lon_max) in JURISDICTION_BBOX.items():
        if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max:
            return name
    return "Other"

smap_all["county"] = smap_all.apply(
    lambda r: assign_county(r["lat"], r["lon"]), axis=1
)

print("Pixel counts by jurisdiction:")
print(smap_all["county"].value_counts().to_string())

### Cell 4 — Aggregate to Annual Mean per County → `smap_df`

The export script plots one line per county over time.  
We aggregate the pixel-level data to a single mean per county per year.

In [ ]:
smap_df = (
    smap_all
    .groupby(["date", "year", "county"], as_index=False)
    .agg(
        sm_mean_m3m3 = ("sm_mean_m3m3", "mean"),
        valid_days   = ("valid_days",   "mean"),
        n_pixels     = ("sm_mean_m3m3", "count"),
    )
)

# Drop 'Other' pixels that fall outside all defined jurisdictions
smap_df = smap_df[smap_df["county"] != "Other"].copy()

print(f"smap_df shape : {smap_df.shape}")
print(f"Columns       : {list(smap_df.columns)}")
print(f"Years covered : {sorted(smap_df['year'].unique())}")
smap_df.head(10)

---
## Part B — Build `canopy_df`

### Cell 5 — Load the CHM Centroid CSV

`all_2016_centroids.csv` has columns: `tile_id, easting_m, northing_m, height_m`  
(coordinates in EPSG:2284 US survey feet — converted to metres by the fixed pipeline).  
We reproject to WGS84 so we can assign each crown to a jurisdiction.

In [ ]:
local_centroids = f"{WORK_DIR}/all_2016_centroids.csv"
s3.download_file(BUCKET, CENTROID_S3_KEY, local_centroids)

crowns = pd.read_csv(local_centroids)
print(f"Loaded {len(crowns):,} tree crown centroids.")
print(f"Columns : {list(crowns.columns)}")
crowns.head()

### Cell 6 — Reproject Crown Centroids to WGS84 and Assign Jurisdiction

In [ ]:
# The fixed pipeline already converted ft→m, so easting_m / northing_m are
# in metres in EPSG:6591 (NAD83(2011) / Virginia State Plane North, metres).
# If your centroid CSV still has raw feet values, multiply by FT_TO_M first:
#   crowns["easting_m"]  *= FT_TO_M
#   crowns["northing_m"] *= FT_TO_M

transformer = Transformer.from_crs("EPSG:6591", WGS84_CRS, always_xy=True)

lons, lats = transformer.transform(
    crowns["easting_m"].values,
    crowns["northing_m"].values,
)
crowns["lon"] = lons
crowns["lat"] = lats

crowns["jurisdiction"] = crowns.apply(
    lambda r: assign_county(r["lat"], r["lon"]), axis=1
)

print("Crown counts by jurisdiction:")
print(crowns["jurisdiction"].value_counts().to_string())

### Cell 7 — Compute Canopy Cover % from CHM TIF Files

Canopy cover % = (pixels with CHM height ≥ 2 m) / (total valid pixels) × 100.  
We compute this for each `*_chm.tif` tile and aggregate by jurisdiction.

In [ ]:
MIN_CANOPY_HEIGHT_M = 2.0   # metres — consistent with the processing pipeline

# List all CHM TIF objects in S3
paginator = s3.get_paginator("list_objects_v2")
chm_keys  = [
    obj["Key"]
    for page in paginator.paginate(Bucket=BUCKET, Prefix=CHM_S3_PREFIX)
    for obj in page.get("Contents", [])
    if obj["Key"].endswith("_chm.tif")
]
print(f"Found {len(chm_keys)} CHM TIF files in s3://{BUCKET}/{CHM_S3_PREFIX}")

tile_stats = []

for key in chm_keys:
    tile_name = os.path.basename(key).replace("_chm.tif", "")
    local_tif = f"{WORK_DIR}/{os.path.basename(key)}"

    s3.download_file(BUCKET, key, local_tif)

    with rasterio.open(local_tif) as src:
        data    = src.read(1).astype(float)
        nodata  = src.nodata
        # Tile centre in the raster CRS
        cx = (src.bounds.left + src.bounds.right)  / 2
        cy = (src.bounds.bottom + src.bounds.top) / 2
        crs_str = src.crs.to_string() if src.crs else "EPSG:6591"

    # Mask nodata
    if nodata is not None:
        valid_mask = data != nodata
    else:
        valid_mask = np.isfinite(data)

    total_valid  = int(valid_mask.sum())
    canopy_pixels = int((valid_mask & (data >= MIN_CANOPY_HEIGHT_M)).sum())
    canopy_pct    = (canopy_pixels / total_valid * 100) if total_valid > 0 else 0.0

    # Reproject tile centre to WGS84 for jurisdiction assignment
    t2 = Transformer.from_crs(crs_str, WGS84_CRS, always_xy=True)
    tile_lon, tile_lat = t2.transform(cx, cy)
    jurisdiction = assign_county(tile_lat, tile_lon)

    tile_stats.append({
        "tile_name"      : tile_name,
        "jurisdiction"   : jurisdiction,
        "total_pixels"   : total_valid,
        "canopy_pixels"  : canopy_pixels,
        "canopy_pct_2016": round(canopy_pct, 2),
    })

    # Clean up local TIF to save disk space
    os.remove(local_tif)

tile_df = pd.DataFrame(tile_stats)
print(f"\nProcessed {len(tile_df)} CHM tiles.")
tile_df.head()

### Cell 8 — Aggregate to Jurisdiction Level → `canopy_df`

The export script expects columns: `jurisdiction`, `canopy_2015_pct`, `canopy_2020_pct`.  
We have 2016 data from the CHM tiles. If you later process 2020 tiles, add a second column.  
For now we populate `canopy_2015_pct` with the 2016 values (closest available year)  
and leave `canopy_2020_pct` as NaN until 2020 tiles are processed.

In [ ]:
canopy_df = (
    tile_df[tile_df["jurisdiction"] != "Other"]
    .groupby("jurisdiction", as_index=False)
    .agg(
        total_pixels  = ("total_pixels",   "sum"),
        canopy_pixels = ("canopy_pixels",  "sum"),
    )
)

canopy_df["canopy_2015_pct"] = (
    canopy_df["canopy_pixels"] / canopy_df["total_pixels"] * 100
).round(2)

# Placeholder for 2020 data — replace with real values when available
canopy_df["canopy_2020_pct"] = float("nan")

print(f"canopy_df shape : {canopy_df.shape}")
print(f"Columns         : {list(canopy_df.columns)}")
canopy_df

---
## Part C — Verify Both DataFrames Are Ready

### Cell 9 — Quick Sanity Check

In [ ]:
print("── smap_df ──────────────────────────────────────")
print(f"  Shape   : {smap_df.shape}")
print(f"  Columns : {list(smap_df.columns)}")
print(f"  Years   : {sorted(smap_df['year'].unique())}")
print(f"  Counties: {sorted(smap_df['county'].unique())}")
print()
print("── canopy_df ────────────────────────────────────")
print(f"  Shape   : {canopy_df.shape}")
print(f"  Columns : {list(canopy_df.columns)}")
print()
print("Both DataFrames are ready for 1_sagemaker_export.py.")

---
## Part D — Generate and Upload Charts (from `1_sagemaker_export.py`)

### Cell 10 — Install `kaleido` (needed for PNG export)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "kaleido"])
print("kaleido installed.")

### Cell 11 — Export Charts, CSVs, and Upload to S3

In [ ]:
import json
import plotly.express as px
import plotly.io as pio

OUTPUT_DIR = "/home/sagemaker-user/dashboard_exports"
S3_EXPORT_PREFIX = "dashboard-data/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Chart A: SMAP Soil Moisture Time Series ───────────────────────────────────
fig_smap = px.line(
    smap_df,
    x     = "date",
    y     = "sm_mean_m3m3",
    color = "county",
    title = "Annual Mean Soil Moisture 2015–2023",
    labels = {"sm_mean_m3m3": "Soil Moisture (m³/m³)", "date": "Year"},
)
pio.write_json(fig_smap,    f"{OUTPUT_DIR}/smap_timeseries.json")
fig_smap.write_image(       f"{OUTPUT_DIR}/smap_timeseries.png", width=900, height=450)
fig_smap.show()
print("Chart A saved.")

# ── Chart B: Canopy Cover Comparison ─────────────────────────────────────────
fig_canopy = px.bar(
    canopy_df,
    x       = "jurisdiction",
    y       = ["canopy_2015_pct", "canopy_2020_pct"],
    barmode = "group",
    title   = "Tree Canopy Cover (%) — 2015/2016 vs 2020",
    labels  = {"value": "Canopy Cover (%)", "variable": "Year"},
)
pio.write_json(fig_canopy,  f"{OUTPUT_DIR}/canopy_cover_bar.json")
fig_canopy.write_image(     f"{OUTPUT_DIR}/canopy_cover_bar.png", width=900, height=450)
fig_canopy.show()
print("Chart B saved.")

# ── Export CSVs and JSONs ─────────────────────────────────────────────────────
smap_df.to_csv(   f"{OUTPUT_DIR}/smap_annual_means.csv",  index=False)
smap_df.to_json(  f"{OUTPUT_DIR}/smap_annual_means.json", orient="records", indent=2)
canopy_df.to_csv( f"{OUTPUT_DIR}/canopy_summary.csv",     index=False)
canopy_df.to_json(f"{OUTPUT_DIR}/canopy_summary.json",    orient="records", indent=2)

metadata = {
    "project_title": "Central Virginia Tree Canopy Change Detection",
    "last_updated"  : pd.Timestamp.now().strftime("%Y-%m-%d"),
    "smap_years"    : sorted(smap_df["year"].unique().tolist()),
    "jurisdictions" : sorted(canopy_df["jurisdiction"].tolist()),
}
with open(f"{OUTPUT_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# ── Upload everything to S3 ───────────────────────────────────────────────────
uploaded = []
for fname in os.listdir(OUTPUT_DIR):
    fpath  = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        s3_key = S3_EXPORT_PREFIX + fname
        s3.upload_file(fpath, BUCKET, s3_key)
        uploaded.append(f"s3://{BUCKET}/{s3_key}")
        print(f"  Uploaded: s3://{BUCKET}/{s3_key}")

print(f"\nAll exports complete. {len(uploaded)} files uploaded to S3.")